# Feature Engineering — Phase 1

This notebook prepare ML-ready feature parquets. 
Run this **before** `2x_starting_kit_xxx` notebooks.

**Features computed**:
1. Wind speed and direction from u/v components (10m and 100m)
2. Sub-daily features (06/12/18 UTC values as extra columns)
3. Lag features (0, 1, 3, 7 days) for key variables
4. Rolling statistics (3-day, 7-day) for wind speed
5. Temporal encodings (month, week-of-year — cyclical)
6. elevation
7. HRES NWP surface forecasts (speed/direction at d1/d7/d10) + `has_hres` indicator
8. HRES NWP pressure-level forecasts (u/v at 1000/925/850/700/500 hPa, d1/d7/d10)

## 1. Setup

In [1]:
import os, sys
# Ensure we're in the notebook's directory (participant_kit/phase1/)
_this_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else None
if _this_dir is None:
    for _candidate in [os.getcwd(), os.path.join(os.getcwd(), "participant_kit", "phase1")]:
        if os.path.exists(os.path.join(_candidate, "utils.py")):
            os.chdir(_candidate)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# -- Paths (adjust DATA_DIR to where you unzipped the dataset) --
DATA_DIR = Path("phase1_dataset")
TRAIN_DIR = DATA_DIR / "train"
INFERENCE_DIR = DATA_DIR / "inference"
FEATURES_DIR = DATA_DIR / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = ["north_sea", "east_china_sea"]
NUM_WINDOWS = 8

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {FEATURES_DIR.resolve()}")
print(f"Regions: {REGIONS}")
print(f"Inference windows: 1-{NUM_WINDOWS}")

# Validate input files exist
print("Checking input data...")
for region in REGIONS:
    for fname in [f"reanalysis_{region}_6h.parquet", f"hres_{region}.parquet",
                  f"elevation_{region}.nc"]:
        fpath = TRAIN_DIR / fname
        assert fpath.exists(), f"Missing: {fpath}. Download the dataset first."
    print(f"  {region}: all input files found")
print("Input validation passed.")


Data directory: /Users/david/projects/hackathon_2026/wind-prediction-private/participant_kit/phase1/phase1_dataset
Output directory: /Users/david/projects/hackathon_2026/wind-prediction-private/participant_kit/phase1/phase1_dataset/features
Regions: ['north_sea', 'east_china_sea']
Inference windows: 1-8
Checking input data...
  north_sea: all input files found
  east_china_sea: all input files found
Input validation passed.


In [ ]:
from feature_engineering import (
    compute_wind_speed_direction, add_lag_features, add_rolling_features,
    add_temporal_features, add_elevation, add_hres_features,
    add_hres_pressure_features,
    pivot_subdaily_features, build_features, drop_redundant_features,
    load_worldwide_data, load_worldwide_context,
    compute_teleconnection_proxies, compute_mslp_pca,
    compute_upstream_features, build_worldwide_features, merge_worldwide_features,
    build_inference_features, compute_vertical_ratios,
)

print(f"Loaded {18} feature engineering functions from feature_engineering.py")


## 2. Process Training Data

Load raw reanalysis for each region, apply the full feature pipeline, merge worldwide features, and save as parquet.


In [3]:
# Pre-compute worldwide features (shared across regions)
worldwide_feats = build_worldwide_features(TRAIN_DIR, years=[2019, 2020, 2021])

Loading worldwide reanalysis data...
  Loaded reanalysis_worldwide_daily_2019.parquet: 23,783,400 rows
  Loaded reanalysis_worldwide_daily_2020.parquet: 23,848,560 rows
  Loaded reanalysis_worldwide_daily_2021.parquet: 23,783,400 rows

Computing teleconnection proxies...
  Teleconnection proxies: 1096 days, NAO range [-5.37, 4.65]

Computing MSLP PCA...
  natl PCA: 6 components explain 68.3% variance (7381 grid points)
  wpac PCA: 6 components explain 80.3% variance (4590 grid points)

Computing upstream features...
  north_sea upstream features: 40 columns (5 points × 2 vars × 4 lags)
  east_china_sea upstream features: 40 columns (5 points × 2 vars × 4 lags)


In [4]:
for region in REGIONS:
    print(f"\n{'='*60}")
    print(f"Processing training data: {region}")
    print(f"{'='*60}")

    # Load raw reanalysis
    reanalysis_path = TRAIN_DIR / f"reanalysis_{region}_6h.parquet"
    df_raw = pd.read_parquet(reanalysis_path)
    df_raw["time"] = pd.to_datetime(df_raw["time"])
    print(f"  Raw shape: {df_raw.shape}")
    print(f"  Time range: {df_raw['time'].min()} to {df_raw['time'].max()}")
    print(f"  Grid points: {df_raw.groupby(['latitude', 'longitude']).ngroups}")

    # Apply feature engineering (daily pivot + lags + HRES etc.)
    df_feat = build_features(df_raw, region, TRAIN_DIR)
    print(f"  Feature-enriched shape: {df_feat.shape}")

    # Merge worldwide features (teleconnections, MSLP PCA, upstream)
    df_feat = merge_worldwide_features(df_feat, worldwide_feats, region)
    print(f"  After worldwide merge: {df_feat.shape}")

    # NaN summary
    nan_pct = df_feat.isna().mean() * 100
    nan_cols = nan_pct[nan_pct > 0].sort_values(ascending=False)
    if len(nan_cols) > 0:
        print(f"  Columns with NaN ({len(nan_cols)}):")
        for col, pct in nan_cols.head(10).items():
            print(f"    {col}: {pct:.1f}%")
        if len(nan_cols) > 10:
            print(f"    ... and {len(nan_cols) - 10} more")
    else:
        print(f"  No NaN columns")

    # Save
    out_path = FEATURES_DIR / f"train_{region}.parquet"
    df_feat.to_parquet(out_path, index=False)
    size_mb = out_path.stat().st_size / 1024**2
    print(f"  Saved: {out_path} ({size_mb:.1f} MB)")

print("\nTraining data processing complete.")



Processing training data: north_sea
  Raw shape: (11244960, 39)
  Time range: 2019-01-01 00:00:00 to 2021-12-31 18:00:00
  Grid points: 2565
  After daily pivot: 2,811,240 rows (was 11,244,960)
  HRES: merged 24 forecast columns, 2,811,240/2,811,240 rows matched (100%)
  Dropped 21 redundant features: ['has_hres', 'msl_h12', 'msl_lag0d', 'sshf_lag0d', 't2m_h12', 't2m_lag0d', 'u10', 'u100', 'v10', 'v100', 'wd10_h12', 'wd10_lag0d', 'ws10_daily_min', 'ws10_h12', 'ws10_lag0d', 'z700_lag0d', 'z850', 'z850_lag0d', 'z850_lag1d', 'z850_lag3d', 'z850_lag7d']
  Feature-enriched shape: (2811240, 99)
  Merged 51 worldwide features for north_sea
  After worldwide merge: (2811240, 150)
  Columns with NaN (59):
    fcst_speed_d10_h6: 66.7%
    fcst_dir_d10_h6: 66.7%
    fcst_dir_d10_h0: 66.7%
    fcst_speed_d10_h0: 66.7%
    fcst_speed_d10_h12: 66.7%
    fcst_dir_d10_h12: 66.7%
    fcst_speed_d10_h18: 66.7%
    fcst_dir_d10_h18: 66.7%
    sst: 38.5%
    msl_lag7d: 0.6%
    ... and 49 more
  Saved: p

## 3. Process Inference Windows

For each of the 8 inference windows, load context reanalysis + HRES, apply the same feature pipeline, and keep only the **last day** of context (where all lag features are fully populated).


In [ ]:
# Process inference windows 
print("Loading training worldwide reanalysis (2019-2021) once...")
ww_train = load_worldwide_data(TRAIN_DIR, years=[2019, 2020, 2021])

for wid in range(1, NUM_WINDOWS + 1):
    window_dir = INFERENCE_DIR / f"window_{wid}"
    if not window_dir.exists():
        print(f"\nWindow {wid}: not found, skipping")
        continue

    print(f"\n{'='*60}\nWindow {wid}: building worldwide features\n{'='*60}")

    # Load this window's worldwide context (≤ context_end, no leakage)
    ww_ctx = load_worldwide_context(window_dir)
    if ww_ctx is None:
        print(f"  WARNING: no context_worldwide_daily.parquet in {window_dir}; "
              f"inference features will lack worldwide columns")
        window_ww_feats = {}
    else:
        ww_combined = pd.concat([ww_train, ww_ctx], ignore_index=True)
        ww_combined = ww_combined.sort_values("time").reset_index(drop=True)

        print(f"  Combined worldwide rows: {len(ww_combined):,} "
              f"({ww_combined['time'].min().date()} → {ww_combined['time'].max().date()})")

        window_ww_feats = {}
        window_ww_feats["teleconnections"] = compute_teleconnection_proxies(ww_combined)
        pca_na, _ = compute_mslp_pca(ww_combined, "natl",
                                     lat_range=(20, 80), lon_range=(-80, 40), n_components=6)
        window_ww_feats["pca_north_atlantic"] = pca_na
        pca_wp, _ = compute_mslp_pca(ww_combined, "wpac",
                                     lat_range=(10, 60), lon_range=(90, 180), n_components=6)
        window_ww_feats["pca_west_pacific"] = pca_wp
        window_ww_feats["upstream_north_sea"] = compute_upstream_features(ww_combined, "north_sea")
        window_ww_feats["upstream_ecs"] = compute_upstream_features(ww_combined, "east_china_sea")

        del ww_combined, ww_ctx

    for region in REGIONS:
        df_inf = build_inference_features(window_dir, region, TRAIN_DIR)
        if df_inf is None:
            print(f"  Window {wid}, {region}: no context reanalysis, skipping")
            continue

        if window_ww_feats:
            df_inf = merge_worldwide_features(df_inf, window_ww_feats, region)

        out_path = FEATURES_DIR / f"inference_window_{wid}_{region}.parquet"
        df_inf.to_parquet(out_path, index=False)
        size_kb = out_path.stat().st_size / 1024
        print(f"  Window {wid}, {region}: {df_inf.shape} -> {out_path.name} ({size_kb:.0f} KB)")

del ww_train
print("\nInference window processing complete.")


Loading 2022 worldwide data for inference...
  2022 worldwide: 23,783,400 rows
  Computing 2022 teleconnection proxies...
  Teleconnection proxies: 365 days, NAO range [-5.69, 5.05]
  Computing 2022 MSLP PCA...
  natl PCA: 6 components explain 69.7% variance (7381 grid points)
  wpac PCA: 6 components explain 83.3% variance (4590 grid points)
  Computing 2022 upstream features...
  north_sea upstream features: 40 columns (5 points × 2 vars × 4 lags)
  east_china_sea upstream features: 40 columns (5 points × 2 vars × 4 lags)
  Merged 51 worldwide features for north_sea
  Window 1, north_sea: (2565, 147) -> inference_window_1_north_sea.parquet (1166 KB)
  Merged 51 worldwide features for east_china_sea
  Window 1, east_china_sea: (2565, 147) -> inference_window_1_east_china_sea.parquet (1170 KB)
  Merged 51 worldwide features for north_sea
  Window 2, north_sea: (2565, 147) -> inference_window_2_north_sea.parquet (1176 KB)
  Merged 51 worldwide features for east_china_sea
  Window 2, eas

## 4. Vertical Wind Profile Ratios

Compute monthly ws(level)/ws(10m) ratios per grid point from reanalysis pressure data.
Used by `2a_starting_kit_light.ipynb` to extend 10m predictions to all vertical levels (100m, 1000-500 hPa).


In [6]:
for region in REGIONS:
    print(f"\n{region}:")
    ratios = compute_vertical_ratios(TRAIN_DIR, region)
    if ratios is not None:
        out_path = FEATURES_DIR / f"vertical_ratios_{region}.parquet"
        ratios.to_parquet(out_path, index=False)
        size_kb = out_path.stat().st_size / 1024
        print(f"  Saved: {out_path.name} ({size_kb:.0f} KB)")

print("\nVertical ratios computed.")



north_sea:
  Loading pressure data: reanalysis_pressure_north_sea.parquet
  Added 30,780 100m ratio records
  Computed 184,680 vertical ratio records (6 levels, 2565 grid points)
  Saved: vertical_ratios_north_sea.parquet (2204 KB)

east_china_sea:
  Loading pressure data: reanalysis_pressure_east_china_sea.parquet
  Added 30,780 100m ratio records
  Computed 184,680 vertical ratio records (6 levels, 2565 grid points)
  Saved: vertical_ratios_east_china_sea.parquet (2207 KB)

Vertical ratios computed.


## 5. Summary

In [7]:
# List all output files with sizes
print(f"Output files in {FEATURES_DIR}/")
print("-" * 60)
total_mb = 0
for p in sorted(FEATURES_DIR.glob("*.parquet")):
    size_mb = p.stat().st_size / 1024**2
    total_mb += size_mb
    nrows = len(pd.read_parquet(p, columns=["latitude"]))
    print(f"  {p.name:<50} {size_mb:>7.1f} MB  ({nrows:>10,} rows)")
print("-" * 60)
print(f"  {'Total':<50} {total_mb:>7.1f} MB")
print(f"\nReady to use in 2a_starting_kit_light.ipynb!")


Output files in phase1_dataset/features/
------------------------------------------------------------
  inference_window_1_east_china_sea.parquet              1.1 MB  (     2,565 rows)
  inference_window_1_north_sea.parquet                   1.1 MB  (     2,565 rows)
  inference_window_2_east_china_sea.parquet              1.1 MB  (     2,565 rows)
  inference_window_2_north_sea.parquet                   1.1 MB  (     2,565 rows)
  inference_window_3_east_china_sea.parquet              1.1 MB  (     2,565 rows)
  inference_window_3_north_sea.parquet                   1.1 MB  (     2,565 rows)
  inference_window_4_east_china_sea.parquet              1.1 MB  (     2,565 rows)
  inference_window_4_north_sea.parquet                   1.1 MB  (     2,565 rows)
  inference_window_5_east_china_sea.parquet              1.1 MB  (     2,565 rows)
  inference_window_5_north_sea.parquet                   1.1 MB  (     2,565 rows)
  inference_window_6_east_china_sea.parquet              1.1 MB  (  